# DecompGTI Cloud Benchmark — Baseline Models

Evaluates baseline instruction-tuned models using the **exact same pipeline** as `scripts/evaluate.py`:

1. Model generates a **structured JSON** with three steps: graph extraction → tool selection → parameters.
2. The extracted graph + parameters are fed into the actual graph algorithm tools (`graph_tools.py`).
3. Metrics match exactly what is reported for the fine-tuned Qwen model:

| Metric | Description |
|---|---|
| `json_validity_rate` | % outputs that are valid JSON with required keys |
| `tool_identification_accuracy` | % correct tool selected |
| `parameter_precision/recall/f1` | Parameter extraction quality |
| `adjacency_extraction_f1` | Graph structure extraction quality |
| `task_success_rate` | % where executed tool gives correct answer |

**Paths assume notebook runs from the project root** (same directory as `src/`, `data/`, `scripts/`).

In [1]:
!pip install -q transformers accelerate bitsandbytes peft datasets tqdm huggingface_hub

In [2]:
import sys, os, gc, json, time, re, importlib.util, torch
from pathlib import Path
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ── Project root (notebook lives in root) ──────────────────────────────────
PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'graph_tools path: {PROJECT_ROOT / "src" / "decompgti" / "mcp_server" / "graph_tools.py"}')

Project root: /home/ramji.purwar/DecompGTI/DecompGTI
graph_tools path: /home/ramji.purwar/DecompGTI/DecompGTI/src/decompgti/mcp_server/graph_tools.py


In [ ]:
from huggingface_hub import login
import os
login(token=os.environ.get('HF_TOKEN', ''))

## Load evaluation metrics (same as `evaluation/metrics.py`)

In [4]:
# Import the same metrics module used by scripts/evaluate.py
from evaluation.metrics import (
    EvalResult,
    aggregate_results,
    check_json_validity,
    check_tool_accuracy,
    check_parameter_extraction,
    check_adjacency_extraction,
    check_task_success,
)
print('✅ Metrics loaded from evaluation/metrics.py')

✅ Metrics loaded from evaluation/metrics.py


In [5]:
# Load graph_tools.py — same loader as scripts/evaluate.py
_spec = importlib.util.spec_from_file_location(
    'graph_tools',
    PROJECT_ROOT / 'src' / 'decompgti' / 'mcp_server' / 'graph_tools.py',
)
graph_tools = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(graph_tools)
print('✅ graph_tools loaded')

✅ graph_tools loaded


## Tool dispatch + execution (identical to `scripts/evaluate.py`)

In [6]:
# ── Exact copy of TOOL_DISPATCH and execute_tool from scripts/evaluate.py ──

TOOL_DISPATCH = {
    'shortest_path': 'dijkstra_shortest_path',
    'dijkstra_shortest_path': 'dijkstra_shortest_path',
    'depth_first_search': 'dfs', 'dfs': 'dfs',
    'breadth_first_search': 'bfs', 'bfs': 'bfs',
    'detect_cycle': 'cycle_detection',
    'cycle_detection': 'cycle_detection', 'cycle': 'cycle_detection',
    'check_connectivity': 'check_connectivity', 'connectivity': 'check_connectivity',
    'topological_sort': 'topological_sort',
    'check_bipartite': None, 'bipartite': None,
    'minimum_spanning_tree': 'minimum_spanning_tree',
    'mst': 'minimum_spanning_tree',
    'maximum_flow': 'maximum_flow', 'max_flow': 'maximum_flow',
    'find_connected_components': None,
    'connected_components': None, 'connected_component': None,
}

TOOL_FUNCTIONS = {
    'dijkstra_shortest_path': graph_tools.dijkstra_shortest_path,
    'bfs':                    graph_tools.bfs,
    'dfs':                    graph_tools.dfs,
    'maximum_flow':           graph_tools.maximum_flow,
    'topological_sort':       graph_tools.topological_sort,
    'cycle_detection':        graph_tools.cycle_detection,
    'minimum_spanning_tree':  graph_tools.minimum_spanning_tree,
    'check_connectivity':     graph_tools.check_connectivity,
}


def parse_adjacency_to_edges(adj_str, directed):
    """Handle multiple adjacency list formats baseline models may produce."""
    edges, seen = [], set()

    # If it's not a string (e.g. Mistral returned a list/dict), skip
    if not isinstance(adj_str, str):
        return edges

    node_pat = re.compile(r'(\d+)\s*:\s*\[(.*?)\]')

    # Match BOTH "weight:5" AND plain "(neighbor, weight)" i.e. "(2, 7)"
    edge_pat = re.compile(r'\((\d+),\s*(?:weight\s*:\s*)?(\d+)\)')

    for nm in node_pat.finditer(adj_str):
        u = int(nm.group(1))
        for em in edge_pat.finditer(nm.group(2)):
            v, w = int(em.group(1)), int(em.group(2))
            if directed:
                edges.append([u, v, w])
            else:
                key = (min(u, v), max(u, v))
                if key not in seen:
                    seen.add(key)
                    edges.append([u, v, w])
    return edges


def normalize_params(params):
    """Map alternative parameter key names to what execute_tool expects."""
    key_aliases = {
        'start_node':   'source',
        'start':        'source',
        'from':         'source',
        'from_node':    'source',
        'end_node':     'target',
        'end':          'target',
        'to':           'target',
        'to_node':      'target',
        'destination':  'target',
        'src':          'source',
        'dst':          'target',
    }
    normalized = {}
    for k, v in params.items():
        canonical = key_aliases.get(k.lower(), k.lower())
        normalized[canonical] = v
    return normalized


def execute_tool(parsed_output):
    tool_name  = parsed_output.get('step2_tool_name', '')
    params     = parsed_output.get('step3_tool_parameters', {})  
    params     = normalize_params(params) 
    graph_info = parsed_output.get('step1_graph_extraction', {})
    adj_str    = graph_info.get('adjacency_list', '')
    directed   = graph_info.get('directed', False)

    dispatch_name = TOOL_DISPATCH.get(tool_name.lower())
    if dispatch_name is None or dispatch_name not in TOOL_FUNCTIONS:
        return None

    tool_fn = TOOL_FUNCTIONS[dispatch_name]
    edges   = parse_adjacency_to_edges(adj_str, directed)

    try:
        kwargs = {'edges': edges}
        if dispatch_name == 'dijkstra_shortest_path':
            kwargs.update(source=params.get('source'), target=params.get('target'), directed=directed)
        elif dispatch_name in ('bfs', 'dfs'):
            kwargs.update(source=params.get('source'), directed=directed)
        elif dispatch_name == 'maximum_flow':
            kwargs.update(source=params.get('source'), target=params.get('target'))
        elif dispatch_name == 'cycle_detection':
            kwargs['directed'] = directed
        elif dispatch_name == 'check_connectivity':
            kwargs.update(source=params.get('source'), target=params.get('target'), directed=directed)

        result = tool_fn(**kwargs)
        for key in ('distance', 'max_flow', 'has_cycle', 'connected', 'total_weight', 'order'):
            if key in result:
                return result[key]
        return result
    except Exception:
        return None


print('✅ Tool dispatch ready.')

✅ Tool dispatch ready.


## Prompt format

Baseline models use `apply_chat_template` with a system prompt that **explicitly instructs the model to produce the same three-step JSON structure** the fine-tuned model was trained on.  
This is the fairest possible comparison — same task, same output format required.

In [7]:
# ── System prompt: tells baseline models to emit the DecompGTI JSON ────────
# Mirrors the INSTRUCTION in scripts/evaluate.py exactly.
SYSTEM_PROMPT = (
    'You are a graph reasoning agent. Given a graph description and a question, '
    'perform these three steps:\n'
    '1. Extract the graph structure as an adjacency list.\n'
    '2. Identify the correct graph algorithm tool to use.\n'
    '3. Extract the parameters required by that tool.\n'
    'When multiple nodes can be visited next in a traversal, always visit the '
    'node with the lowest numerical ID first.\n'
    'Output your answer as a single valid JSON object with exactly these keys:\n'
    '{\n'
    '  "step1_graph_extraction": {\n'
    '    "adjacency_list": "<node: [(neighbor, weight:N), ...], ...>",\n'
    '    "directed": true|false\n'
    '  },\n'
    '  "step2_tool_name": "<tool_name>",\n'
    '  "step3_tool_parameters": { <param_key>: <value>, ... }\n'
    '}\n'
    'Valid tool names: shortest_path, breadth_first_search, depth_first_search, '
    'detect_cycle, check_connectivity, topological_sort, minimum_spanning_tree, maximum_flow.\n'
    'Output ONLY the JSON object. No explanation, no markdown, no extra text.'
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('✅ Config ready.')

✅ Config ready.


## Core evaluation function (mirrors `run_evaluation` in `scripts/evaluate.py`)

In [8]:
def run_inference(model, tokenizer, question_text):
    """Run greedy inference with chat template (no sampling, deterministic)."""
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': question_text},
    ]
    text   = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8192,          # same as scripts/evaluate.py
            do_sample=False,              # greedy — deterministic
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


def evaluate_model_on_testset(model, tokenizer, test_set, model_id, test_path, sample_limit=None):
    """
    Full DecompGTI evaluation pipeline — identical logic to scripts/evaluate.py.
    Returns (metrics_dict, results_list, predictions_list).
    """
    data = test_set[:sample_limit] if sample_limit else test_set
    results, predictions = [], []

    for i, sample in enumerate(tqdm(data, desc=f'{model_id.split("/")[-1]} | {Path(test_path).stem}')):
        question_text = sample['graph_nl'] + '\n\nQuestion: ' + sample['question']

        t0 = time.time()
        raw_output = run_inference(model, tokenizer, question_text)
        elapsed    = time.time() - t0

        predictions.append({
            'sample_id':        sample.get('sample_id', i),
            'raw_output':       raw_output,
            'inference_time_s': round(elapsed, 2),
        })

        # ── Evaluate (same as scripts/evaluate.py run_evaluation) ──────────
        result             = EvalResult()
        result.sample_id   = sample.get('sample_id', i)
        result.task_type   = sample.get('task_type', '')
        result.graph_size  = sample.get('graph_size', '')
        result.num_nodes   = sample.get('num_nodes', 0)

        result.json_valid, parsed = check_json_validity(raw_output)

        if result.json_valid and parsed:
            result.predicted_tool = parsed.get('step2_tool_name', '')
            result.expected_tool  = sample.get('expected_tool', '')
            result.tool_correct   = check_tool_accuracy(result.predicted_tool, result.expected_tool)

            result.predicted_params  = parsed.get('step3_tool_parameters', {})
            result.expected_params   = sample.get('expected_params', {})
            result.params_precision, result.params_recall = check_parameter_extraction(
                result.predicted_params, result.expected_params
            )

            graph_info   = parsed.get('step1_graph_extraction', {})
            pred_adj     = graph_info.get('adjacency_list', '')
            expected_adj = sample.get('graph_adj', '')
            directed     = sample.get('directed', False)
            result.adj_edge_f1 = check_adjacency_extraction(pred_adj, expected_adj, directed=directed)

            try:
                actual_answer   = execute_tool(parsed)
                expected_answer = sample.get('expected_answer')
                result.task_success = check_task_success(actual_answer, expected_answer)
            except Exception:
                result.task_success = False
        else:
            result.error_message = 'Invalid JSON'

        results.append(result)

    metrics = aggregate_results(results)
    return metrics, results, predictions


print('✅ Evaluation functions ready.')

✅ Evaluation functions ready.


---
## 🧪 SAMPLE RUN — 10 samples, verbose debug

**Run this first** to verify JSON output looks correct before committing to full benchmark.

In [9]:
SAMPLE_LIMIT   = 10
SAMPLE_MODEL   = 'mistralai/Mistral-7B-Instruct-v0.3'
SAMPLE_DATASET = 'data/test_set_mini.json'
SAMPLE_OUT_DIR = 'evaluation/results/sample_decompgti'
os.makedirs(SAMPLE_OUT_DIR, exist_ok=True)

with open(SAMPLE_DATASET) as f:
    full_data = json.load(f)

# Pick up to 2 samples per task type for diversity
task_seen, sample_data = {}, []
for item in full_data:
    tt = item.get('task_type', 'unknown')
    task_seen.setdefault(tt, 0)
    if task_seen[tt] < 2:
        sample_data.append(item); task_seen[tt] += 1
    if len(sample_data) >= SAMPLE_LIMIT: break

# Pad to SAMPLE_LIMIT if needed
if len(sample_data) < SAMPLE_LIMIT:
    sel = {id(x) for x in sample_data}
    for item in full_data:
        if id(item) not in sel:
            sample_data.append(item)
        if len(sample_data) >= SAMPLE_LIMIT: break

print(f'📋 {len(sample_data)} samples | tasks covered: {list(task_seen.keys())}')

# Load model
print(f'\n🚀 Loading: {SAMPLE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(SAMPLE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    SAMPLE_MODEL, quantization_config=quantization_config,
    device_map='cuda:1', trust_remote_code=True
)
model.eval()
print(f'✅ Model on: {model.device}')

📋 10 samples | tasks covered: ['shortest_path', 'bfs', 'dfs', 'cycle', 'connectivity']

🚀 Loading: mistralai/Mistral-7B-Instruct-v0.3


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]/home/ramji.purwar/miniconda3/envs/llama_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 4949.45 MB. The target location /home/ramji.purwar/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/blobs only has 405.86 MB free disk space.
  warnings.warn(
/home/ramji.purwar/miniconda3/envs/llama_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 4999.82 MB. The target location /home/ramji.purwar/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/blobs only has 405.86 MB free disk space.
  warnings.warn(
/home/ramji.purwar/miniconda3/envs/llama_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected fi

KeyboardInterrupt: 

In [17]:
# Verbose sample run — inspect every row
DIV = '=' * 72
sample_predictions = []

for i, sample in enumerate(sample_data):
    question_text = sample['graph_nl'] + '\n\nQuestion: ' + sample['question']

    t0 = time.time()
    raw = run_inference(model, tokenizer, question_text)
    elapsed = time.time() - t0

    json_valid, parsed = check_json_validity(raw)

    tool_ok = False
    task_success = False
    adj_f1 = 0.0

    if json_valid and parsed:
        pred_tool    = parsed.get('step2_tool_name', '')
        expected_tool = sample.get('expected_tool', '')
        tool_ok      = check_tool_accuracy(pred_tool, expected_tool)

        graph_info   = parsed.get('step1_graph_extraction', {})
        adj_f1       = check_adjacency_extraction(
            graph_info.get('adjacency_list', ''),
            sample.get('graph_adj', ''),
            directed=sample.get('directed', False)
        )
        try:
            actual   = execute_tool(parsed)
            expected = sample.get('expected_answer')
            task_success = check_task_success(actual, expected)
        except Exception:
            pass

    status = '✅ SUCCESS' if task_success else ('⚠️  TOOL OK' if tool_ok else '❌ FAIL')
    print(f'\n{DIV}')
    print(f'Sample {i+1:02d}/{len(sample_data)}  |  task: {sample.get("task_type","?")}  |  {status}')
    print(f'  Question   : {sample["question"]}')
    print(f'  JSON valid : {json_valid}  |  Tool correct: {tool_ok}  |  Adj F1: {adj_f1:.2f}')
    print(f'  Task success: {task_success}  |  Time: {elapsed:.2f}s')
    print(f'  Raw output (first 400 chars):')
    print(f'    {raw[:400].strip()!r}' + (' [TRUNCATED]' if len(raw) > 400 else ''))

    sample_predictions.append({
        'sample_id':        sample.get('sample_id', i),
        'question':         sample['question'],
        'task_type':        sample.get('task_type', ''),
        'raw_output':       raw,
        'json_valid':       json_valid,
        'tool_correct':     tool_ok,
        'adj_f1':           round(adj_f1, 4),
        'task_success':     task_success,
        'inference_time_s': round(elapsed, 2),
    })

# Save sample predictions
safe_model = SAMPLE_MODEL.split('/')[-1]
with open(f'{SAMPLE_OUT_DIR}/sample_{safe_model}_predictions.json', 'w') as f:
    json.dump(sample_predictions, f, indent=2)
print(f'\n✅ Sample predictions saved → {SAMPLE_OUT_DIR}/')

del model, tokenizer
gc.collect(); torch.cuda.empty_cache()
print('🧹 GPU cleared.')


Sample 01/10  |  task: shortest_path  |  ⚠️  TOOL OK
  Question   : Calculate the distance of the shortest path from node 1 to node 6.
  JSON valid : True  |  Tool correct: True  |  Adj F1: 0.00
  Task success: False  |  Time: 32.86s
  Raw output (first 400 chars):
    '{\n  "step1_graph_extraction": {\n    "adjacency_list": [\n      { "0": [{"neighbor": 2, "weight": 7}, {"neighbor": 3, "weight": 5}, {"neighbor": 4, "weight": 8}], "1": false },\n      { "2": [{"neighbor": 5, "weight": 6}], "2": false },\n      { "3": [{"neighbor": 1, "weight": 3}, {"neighbor": 4, "weight": 6}, {"neighbor": 5, "weight": 6}], "3": false },\n      { "4": [{"neighbor": 0, "weight": 4}, {"n' [TRUNCATED]

Sample 02/10  |  task: shortest_path  |  ✅ SUCCESS
  Question   : Calculate the distance of the shortest path from node 1 to node 5.
  JSON valid : True  |  Tool correct: True  |  Adj F1: 0.00
  Task success: True  |  Time: 16.38s
  Raw output (first 400 chars):
    '{\n  "step1_graph_extraction": {\n    "

---
## 🚀 FULL BENCHMARK

Runs every model × every test set. Produces output files in the same format as `scripts/evaluate.py`:
- `evaluation/results/baseline_{model}_{testset}.json` — metrics
- `evaluation/results/baseline_{model}_{testset}_predictions.json` — raw predictions

In [11]:
# ── Configuration ──────────────────────────────────────────────────────────
TEST_SETS = [
    'data/test_set_mini.json',
    'data/test_set_small.json',
    'data/test_set_medium.json',
    'data/test_set_large.json',
]

MODELS = [
    # 'mistralai/Mistral-7B-Instruct-v0.3',
    # 'meta-llama/Meta-Llama-3-8B-Instruct',
    # 'Qwen/Qwen2.5-7B-Instruct',
    # 'microsoft/phi-4-mini-instruct'
    'Qwen/Qwen3-8B'
]

# Set to None to run the full test set, or an int to cap for quick testing
SAMPLE_LIMIT = 100

os.makedirs('evaluation/results', exist_ok=True)
print(f'Models to evaluate: {MODELS}')
print(f'Test sets: {TEST_SETS}')
print(f'Sample limit: {SAMPLE_LIMIT or "full"}')

Models to evaluate: ['Qwen/Qwen3-8B']
Test sets: ['data/test_set_mini.json', 'data/test_set_small.json', 'data/test_set_medium.json', 'data/test_set_large.json']
Sample limit: 100


In [12]:
all_summary = {}   # model_id → {test_set_name → metrics_dict}

for model_id in MODELS:
    print(f"\n{'='*70}")
    print(f"🚀 {model_id}")
    print(f"{'='*70}")

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=quantization_config,
            device_map='cuda:1',
            trust_remote_code=True,
        )
        model.eval()
        print(f'✅ Loaded — device: {model.device}')

        safe_model = model_id.split('/')[-1]
        all_summary[model_id] = {}

        for test_path in TEST_SETS:
            if not os.path.exists(test_path):
                print(f'  ⚠️  Skipping (not found): {test_path}')
                continue

            with open(test_path) as f:
                test_set = json.load(f)

            set_name = Path(test_path).stem.replace('test_set_', '')
            n_samples = len(test_set[:SAMPLE_LIMIT]) if SAMPLE_LIMIT else len(test_set)
            print(f'\n  >> {safe_model} on {set_name} ({n_samples} samples)')

            t0 = time.time()
            metrics, results, predictions = evaluate_model_on_testset(
                model, tokenizer, test_set, model_id, test_path,
                sample_limit=SAMPLE_LIMIT,
            )
            elapsed = time.time() - t0

            # Print report in same style as scripts/evaluate.py
            metrics.print_report()
            print(f'  Time: {elapsed:.1f}s ({elapsed/n_samples:.2f}s/sample)')

            # Save results — same structure as scripts/evaluate.py
            metrics_dict = metrics.to_dict()
            result_record = {
                'config': {
                    'base_model':    model_id,
                    'adapter':       None,          # no adapter — baseline
                    'test_set':      test_path,
                    'total_time_s':  round(elapsed, 1),
                    'samples':       n_samples,
                },
                'metrics': metrics_dict,
            }

            out_f = f'evaluation/results/baseline_{safe_model}_{set_name}.json'
            pre_f = f'evaluation/results/baseline_{safe_model}_{set_name}_predictions.json'

            with open(out_f, 'w') as f:
                json.dump(result_record, f, indent=2)
            with open(pre_f, 'w') as f:
                json.dump(predictions, f, indent=2)

            print(f'  ✅ Saved: {out_f}')
            all_summary[model_id][set_name] = metrics_dict

    except Exception as e:
        import traceback
        print(f'❌ ERROR with {model_id}: {e}')
        traceback.print_exc()
    finally:
        try:
            del model, tokenizer
        except NameError:
            pass
        gc.collect()
        torch.cuda.empty_cache()
        time.sleep(2)
        print('  🧹 GPU cleared.')

print(f"\n{'='*70}\n🎉 ALL BENCHMARKS COMPLETE!\n{'='*70}")


🚀 Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

✅ Loaded — device: cuda:1

  >> Qwen3-8B on mini (89 samples)


Qwen3-8B | test_set_mini:   0%|          | 0/89 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  🧹 GPU cleared.


KeyboardInterrupt: 

## Summary table — comparable to fine-tuned Qwen results

In [ ]:
# Print a consolidated comparison table
# Columns match what scripts/evaluate.py prints in its combined summary

HDR = f"  {'Model':<40} {'Set':<8} {'N':>5} {'JSON%':>7} {'Tool%':>7} {'PF1':>7} {'AdjF1':>7} {'Succ%':>7}"
print(f"\n{'='*90}")
print(f"  BASELINE SUMMARY (DecompGTI pipeline)")
print(f"{'='*90}")
print(HDR)
print(f"  {'-'*85}")

for model_id, sets in all_summary.items():
    safe = model_id.split('/')[-1]
    for set_name, m in sets.items():
        print(
            f"  {safe:<40} {set_name:<8} "
            f"{m.get('total_samples',0):>5} "
            f"{m.get('json_validity_rate',0)*100:>6.1f}% "
            f"{m.get('tool_identification_accuracy',0)*100:>6.1f}% "
            f"{m.get('parameter_f1',0)*100:>6.1f}% "
            f"{m.get('adjacency_extraction_f1',0)*100:>6.1f}% "
            f"{m.get('task_success_rate',0)*100:>6.1f}%"
        )

print(f"  {'-'*85}")
print(f"\n  Reference — Fine-tuned Qwen2.5-7B (mini, 373 samples):")
print(f"  {'Qwen2.5-7B-LoRA':<40} {'mini':<8} {'373':>5} {'99.2%':>7} {'99.2%':>7} {'98.1%':>7} {'99.7%':>7} {'87.7%':>7}")
print(f"{'='*90}")

# Save combined summary
with open('evaluation/results/baseline_summary_decompgti.json', 'w') as f:
    json.dump(all_summary, f, indent=2)
print('\n✅ Summary saved → evaluation/results/baseline_summary_decompgti.json')